In [1]:
import pandas as pd

In [2]:
df=pd.read_csv("sentiment_data.csv")
print(df)

                                  text sentiment
0                    I love this movie  positive
1                This movie is amazing  positive
2                       Excellent film  positive
3          I really enjoyed this movie  positive
4             The acting was fantastic  positive
5          What a wonderful experience  positive
6            This product is excellent  positive
7            I am very happy with this  positive
8               The quality is amazing  positive
9                  Absolutely loved it  positive
10           Best movie I have watched  positive
11           The story was interesting  positive
12     Great performance by the actors  positive
13         This is a fantastic product  positive
14               I highly recommend it  positive
15           The service was excellent  positive
16     Very satisfied with my purchase  positive
17            This movie was brilliant  positive
18          Amazing experience overall  positive
19               I l

In [3]:
df.head()

,text,sentiment
0,I love this movie,positive
1,This movie is amazing,positive
2,Excellent film,positive
3,I really enjoyed this movie,positive
4,The acting was fantastic,positive


In [6]:
df.isnull().sum()

text         0
sentiment    0
dtype: int64

In [7]:
df.shape

(40, 2)

In [8]:
print(df.columns)

Index(['text', 'sentiment'], dtype='object')


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   text       40 non-null     object
 1   sentiment  40 non-null     object
dtypes: object(2)
memory usage: 772.0+ bytes


In [10]:
print(df.duplicated().sum())

0


In [11]:
print(df["sentiment"].value_counts())

sentiment
positive    20
negative    20
Name: count, dtype: int64


Text Preprocessing

In [12]:
import nltk
import string

In [ ]:
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")

[nltk_data] Downloading package punkt_tab to C:\Users\ADITYA
[nltk_data]     RAO\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\ADITYA
[nltk_data]     RAO\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [49]:
from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [50]:
stop_words = set(stopwords.words("english"))
lemmatizor = WordNetLemmatizer()

In [51]:
# Preprocess the text

def preprocess(text):

    #lower
    text=text.lower()
    #token
    token=word_tokenize(text)

    # stopwords + punctuation
    clean_tokens=[word for word in token if word not in stop_words and word not in string.punctuation]

    #Lemmatization
    Lemmatized=[lemmatizor.lemmatize(word) 
                for word in clean_tokens
                ]
    #join tokens into sentences
    return " ".join(Lemmatized)


In [52]:
df["Clean_text"]=df["text"].apply(preprocess)

df[["text","Clean_text"]].head()

,text,Clean_text
0,I love this movie,love movie
1,This movie is amazing,movie amazing
2,Excellent film,excellent film
3,I really enjoyed this movie,really enjoyed movie
4,The acting was fantastic,acting fantastic


In [53]:
#input and output
x=df["Clean_text"]
y=df["sentiment"]

In [54]:
#train test split
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(
    x,y,test_size=0.2,random_state=42,stratify=y)  #stratify=y ---> +ve and -ve ka ratio train/test mein maintain krta hai

In [55]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizor=TfidfVectorizer()

In [56]:
x_train_tfidf=vectorizor.fit_transform(x_train)
x_test_tfidf=vectorizor.transform(x_test)


In [57]:
#train the model
from sklearn.linear_model import LogisticRegression


model=LogisticRegression()
model.fit(x_train_tfidf,y_train)

LogisticRegression()

In [58]:
#prediction
y_pred=model.predict(x_test_tfidf)
print("Prediction :",y_pred)

Prediction : ['negative' 'positive' 'negative' 'negative' 'negative' 'positive'
 'negative' 'negative']


In [59]:
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
print("Accuracy_score :\n",accuracy_score(y_test,y_pred))
print("confusion_matrix :\n",confusion_matrix(y_pred,y_test))
print("classification_report :\n",classification_report(y_test,y_pred))

Accuracy_score :
 0.5
confusion_matrix :
 [[3 3]
 [1 1]]
classification_report :
               precision    recall  f1-score   support

    negative       0.50      0.75      0.60         4
    positive       0.50      0.25      0.33         4

    accuracy                           0.50         8
   macro avg       0.50      0.50      0.47         8
weighted avg       0.50      0.50      0.47         8



In [60]:
print("Actual :",y_test)
print("Predicted :",y_pred)

Actual : 27    negative
34    negative
17    positive
11    positive
26    negative
1     positive
30    negative
0     positive
Name: sentiment, dtype: object
Predicted : ['negative' 'positive' 'negative' 'negative' 'negative' 'positive'
 'negative' 'negative']


In [61]:
#new prediction
new_text=["The product quality is amazing and I love it"]
new_text_vec=vectorizor.transform(new_text)
print("Prediction of New_text :",model.predict(new_text_vec))

Prediction of New_text : ['positive']


In [62]:
new_text2=["The service was terrible and I would never recommend to this anymore"]
new_text2_vec=vectorizor.transform(new_text2)
print("Prediction of New_text2 :",model.predict(new_text2_vec))

Prediction of New_text2 : ['negative']
